In [ ]:

import os, sys, glob, shutil, time, json
import numpy as np, pandas as pd
t0=time.perf_counter()
def log(m): print(f"[{time.perf_counter()-t0:6.0f}с] {m}", flush=True)
base=os.path.dirname(glob.glob("/kaggle/input/**/items_human.parquet", recursive=True)[0])
prev=os.path.dirname(glob.glob("/kaggle/input/**/features_human.npy", recursive=True)[0])
os.makedirs("/kaggle/working/src",exist_ok=True)
for p in glob.glob(base+"/*.py"): shutil.copy(p,"/kaggle/working/src/")
open("/kaggle/working/src/__init__.py","a").close()
os.chdir("/kaggle/working"); sys.path.insert(0,"/kaggle/working")
from src.hybrid import product_disjoint_pair_masks
from src.metrics import macro_pr_auc
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import average_precision_score

# Выгрузка деревьев вставлена в ядро, а не берётся из датасета: иначе ради шестидесяти
# строк пришлось бы переливать полгигабайта. Копия совпадает с `src/export_boost.py`.
def export(model):
    F,T,L,R,V,Lf,M,S = [],[],[],[],[],[],[],[]
    off=0
    for stage in model._predictors:
        n=stage[0].nodes
        if n["is_categorical"].any(): raise ValueError("категориальные разбиения не выгружаются")
        S.append(off)
        F.append(n["feature_idx"].astype(np.int32)); T.append(n["num_threshold"].astype(np.float64))
        L.append(n["left"].astype(np.int32)+off);   R.append(n["right"].astype(np.int32)+off)
        V.append(n["value"].astype(np.float64));    Lf.append(n["is_leaf"].astype(np.uint8))
        M.append(n["missing_go_to_left"].astype(np.uint8)); off+=len(n)
    return {"feature":np.concatenate(F),"threshold":np.concatenate(T),"left":np.concatenate(L),
            "right":np.concatenate(R),"value":np.concatenate(V),"is_leaf":np.concatenate(Lf),
            "missing_left":np.concatenate(M),"starts":np.asarray(S,dtype=np.int32),
            "baseline":np.asarray(model._baseline_prediction,dtype=np.float64).ravel()}

def predict_raw(t, X):
    feat,thr,left,right=t["feature"],t["threshold"],t["left"],t["right"]
    leaf,mleft=t["is_leaf"],t["missing_left"]
    total=np.full(len(X),float(t["baseline"][0]),dtype=np.float64)
    for st in t["starts"]:
        node=np.full(len(X),st,dtype=np.int32); active=~leaf[node].astype(bool)
        while active.any():
            rows=np.flatnonzero(active); cur=node[rows]; col=feat[cur]
            val=X[rows,col]; go=val<=thr[cur]
            blank=np.isnan(val)
            if blank.any(): go=np.where(blank,mleft[cur].astype(bool),go)
            node[rows]=np.where(go,left[cur],right[cur]); active[rows]=~leaf[node[rows]].astype(bool)
        total+=t["value"][node]
    return total

def predict_proba(t, X): return 1.0/(1.0+np.exp(-np.clip(predict_raw(t,X),-30.0,30.0)))
def save(path,t): np.savez_compressed(path,**t)


Xh=np.load(prev+"/features_human.npy"); Xl=np.load(prev+"/features_llm.npy")
E1=np.load(prev+"/features_eval_lex.npy"); E2=np.load(prev+"/features_eval_mixed.npy")
items=pd.read_parquet(base+"/items_human.parquet",columns=["id","category"])
li=pd.read_parquet(base+"/llm_items_sel.parquet",columns=["id","category"])
hm=pd.read_parquet(base+"/matches.parquet",columns=["id1","id2","target"])
ev1=pd.read_parquet(base+"/eval_pairs.parquet"); ev2=pd.read_parquet(base+"/eval_pairs_mixed.parquet")
lp=pd.read_parquet(base+"/llm_pairs_sel.parquet")
y=hm["target"].to_numpy(np.int8); yl=lp["label"].to_numpy(np.int8)
cat_of=dict(zip(items["id"],items["category"].astype(str)))
cp=hm["id1"].map(cat_of).astype(str).to_numpy()
tm,vm=product_disjoint_pair_masks(hm["id1"].to_numpy(),hm["id2"].to_numpy(),0,3)
va=np.flatnonzero(vm)
c1=ev1["category"].astype(str).to_numpy(); c2=ev2["category"].astype(str).to_numpy()
y1=ev1["target"].to_numpy(np.int8); y2=ev2["target"].to_numpy(np.int8)
def macro(p,c,yy): return float(np.mean([average_precision_score(yy[c==k],p[c==k])
    for k in np.unique(c) if len(np.unique(yy[c==k]))>1]))

# Боевая модель учится на ВСЕХ доступных парах: holdout нужен только для замера, а в
# решении он не участвует. Признак категории уже посчитан в матрице последним столбцом.
# Проверка гипотезы: признаки окрестности зависят не от пары, а от пула товаров вокруг.
# Обе наши линейки построены на том же пуле, что и обучение, поэтому сдвиг пула на закрытом
# тесте они увидеть не могли. Отправка со всеми 128 признаками дала на лидерборде минус при
# обещанных линейкой плюс 0.033 — здесь те же данные без одиннадцати пул-зависимых столбцов.
from src.attr_features import FEATURE_NAMES
from src.name_features import NAME_FEATURE_NAMES
from src.string_features import STRING_FEATURE_NAMES
from src.neighbour_features import NEIGHBOUR_FEATURE_NAMES
from src.brand_features import BRAND_FEATURE_NAMES
from src.dim_features import DIM_FEATURE_NAMES
ALL=FEATURE_NAMES+NAME_FEATURE_NAMES+STRING_FEATURE_NAMES+NEIGHBOUR_FEATURE_NAMES+BRAND_FEATURE_NAMES+DIM_FEATURE_NAMES
keep=[i for i,n in enumerate(ALL) if n not in set(NEIGHBOUR_FEATURE_NAMES)]
log(f"всего признаков {len(ALL)}, остаётся {len(keep)}, убрано {len(ALL)-len(keep)} (окрестности)")
json.dump({"kept": [ALL[i] for i in keep]}, open("/kaggle/working/kept_features.json","w"), ensure_ascii=False)
Xl=Xl[:,keep]; Xh=Xh[:,keep]; E1=E1[:,keep]; E2=E2[:,keep]
X=np.vstack([Xl,Xh]); yy=np.concatenate([yl,y])
log(f"обучающих пар: {len(X):,} ({len(Xl):,} LLM + {len(Xh):,} ручных), признаков {X.shape[1]}")
# Параметры выбраны по линейке, а не по ручному holdout. Перебор на holdout выводил на
# 1500 деревьев и 255 листьев (+0.0197 там), но на обеих линейках такая модель оказалась
# ХУЖЕ на 0.002-0.004. Holdout нас обманывал уже трижды, поэтому решает линейка.
P=dict(max_iter=800,learning_rate=0.05,max_leaf_nodes=63,
       random_state=0,early_stopping=False)
model=HistGradientBoostingClassifier(**P).fit(X,yy)
log("обучено")

trees=export(model)
save("/kaggle/working/feature_boost.npz",trees)
log(f"выгружено деревьев {len(trees['starts'])}, узлов {len(trees['feature']):,}, "
    f"{os.path.getsize('/kaggle/working/feature_boost.npz')/1e6:.2f} МБ")

# Выгрузка обязана совпадать с исходной моделью: иначе в архив уедет другая модель.
check=np.vstack([E1[:3000],E2[:3000]]).astype(np.float64)
a=model.predict_proba(check)[:,1]; b=predict_proba(trees,check)
log(f"сверка выгрузки: max |разница| {np.abs(a-b).max():.3e}")
if np.abs(a-b).max()>1e-9: raise SystemExit("выгрузка расходится с моделью")

p1=predict_proba(trees,E1.astype(np.float64)); p2=predict_proba(trees,E2.astype(np.float64))
np.save("/kaggle/working/final_pred_lex.npy",p1); np.save("/kaggle/working/final_pred_mix.npy",p2)
log(f"на линейках: лексич {macro(p1,c1,y1):.6f}  смеш {macro(p2,c2,y2):.6f}")
log("ВНИМАНИЕ: holdout не мерим — модель обучена в том числе на нём")
json.dump({"categories":sorted(set(list(cat_of.values())+li["category"].astype(str).tolist())),
           "n_features":int(X.shape[1]),"params":P,"n_train":int(len(X))},
          open("/kaggle/working/final_info.json","w"),ensure_ascii=False,indent=2)
log("готово")
